In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

all_data = []

print(f"\nScraping category: DISTRIBUTOR...\n")

for page in range(1, 20):
    print(f"  Scraping page {page}...")

    url = f"https://ensun.io/search?threshold=VERY_LOW&q=ductile%20iron%20pipe%20fittings&page={page}&locations=United%20States%2Cnull%2Cnull&categories=DISTRIBUTOR"
    driver.get(url)
    time.sleep(3)  # Allow dynamic content to load

    listings = driver.find_elements(By.XPATH, '//div[contains(@class,"MuiPaper-root")]')

    if not listings:
        print("  No listings found, stopping.\n")
        break

    for listing in listings:
        try:
            company_name = listing.find_element(By.XPATH, './/p[contains(@class,"mui-vpiwmb")]').text.strip()
        except:
            company_name = "Not Found"

        try:
            location = listing.find_element(By.XPATH, './/p[contains(@class,"mui-opaoyh")]').text.strip()
        except:
            location = "Not Found"

        try:
            details = listing.find_elements(By.XPATH, './/p[contains(@class,"mui-1k5s1ae")]')
            employees = details[0].text.strip() if len(details) >= 1 else "Not Found"
            founded = details[1].text.strip() if len(details) >= 2 else "Not Found"
        except:
            employees = "Not Found"
            founded = "Not Found"

        try:
            key_descrip = listing.find_element(By.XPATH, './/p[contains(@class,"mui-1tjpl0f")]').text.strip()
        except:
            key_descrip = "Not Found"

        all_data.append({
            "Company Name": company_name,
            "Location": location,
            "Employee Count": employees,
            "Year Founded": founded,
            "Key takeaway": key_descrip
        })

print("\n✅ Finished scraping distributors.")
driver.quit()

# Save to Excel
df = pd.DataFrame(all_data).drop_duplicates()
df


Scraping category: DISTRIBUTOR...

  Scraping page 1...
  Scraping page 2...
  Scraping page 3...
  Scraping page 4...
  Scraping page 5...
  Scraping page 6...
  Scraping page 7...
  Scraping page 8...
  No listings found, stopping.


✅ Finished scraping distributors.


,Company Name,Location,Employee Count,Year Founded,Key takeaway
0,Not Found,Not Found,Not Found,Not Found,Not Found
3,"Eastern Industrial Supplies, Inc.","Greenville, United States",251-500 Employees,1980,"Eastern Industrial Supplies, Inc. specializes ..."
4,Atkore International,"Harvey, United States",1001-5000 Employees,2010,Atkore is a global manufacturer that produces ...
5,SSP Corporation,"Twinsburg, United States",101-250 Employees,1926,SSP is a leading manufacturer of industrial pi...
7,Dakota Supply Group,"Monticello, United States",501-1000 Employees,1898,Dakota Supply Group (DSG) is a comprehensive d...
8,PLIDCO,"Strongsville, United States",101-250 Employees,1949,PLIDCO specializes in high-pressure pipeline r...
9,GPT Industries,"Wheat Ridge, United States",251-500 Employees,2012,GPT Industries specializes in advanced enginee...
10,JB Industries,"Aurora, United States",101-250 Employees,1967,JB Industries is a leading manufacturer of Ame...
11,Ductmate Industries,"Charleroi, United States",101-250 Employees,1978,"Ductmate Industries, Inc. is a prominent manuf..."
12,Eldon James,"Denver, United States",101-250 Employees,1987,Eldon James Corp specializes in manufacturing ...


In [6]:
df.to_excel("USA_Ductile_iron_Pipe_Fitting_Distributor.xlsx", index=False)
print("\n✅ Data saved to 'USA_Ductile_iron_Pipe_Fitting_Distributor.xlsx'")


✅ Data saved to 'USA_Ductile_iron_Pipe_Fitting_Distributor.xlsx'


In [8]:
import pandas as pd
import re
# Load the Excel file
df = pd.ExcelFile('USA_Ductile_iron_Pipe_Fitting_Distributor.xlsx')
data = df.parse('Sheet1')

# Function to identify product type and split location
def process_row(row):
    key_takeaway = str(row['Key takeaway']).lower()
    product_type = 'Yes' if 'ductile iron pipe fittings' in key_takeaway or 'ductile iron' in key_takeaway else 'No'
    location = str(row['Location'])
    if ',' in location:
        city, state = [x.strip() for x in location.split(',', 1)]
    else:
        city, state = '', location.strip()
    
    return pd.Series([product_type, city, state])

# Apply processing
data[['Product Type', 'Location City', 'Location State']] = data.apply(process_row, axis=1)

# Remove the first row (assumed invalid)
data = data.iloc[1:].reset_index(drop=True)

# Clean 'Employee Count' to remove 'Employees'
data['Employee Count'] = data['Employee Count'].str.replace(' Employees', '', regex=False)

# ✅ No filtering - keep all rows

# Select required columns
output_data = data[[
    'Company Name',
    'Location City',
    'Location State',
    'Employee Count',
    'Year Founded',
    'Key takeaway',
    'Product Type'
]]

# Save to Excel
output_data.to_excel('usa_Ductile_iron_Fittings_distributor.xlsx', index=False)
